# Project Delivery Delay Prediction & Root-Cause Analysis

Project delivery delays can occur due to factors such as resource shortages, budget constraints, poor planning, and technical challenges. Identifying potential risks and their root causes early can help organizations take corrective action and improve project delivery. This project uses Large Language Models (LLMs) to generate realistic demonstration project records and analyse each project for potential delivery delays and contributing factors. The generated records are processed in batches, and the same selected LLM performs both data generation and risk analysis. Finally, the results, including risk assessments, root causes, and recommended actions, are exported in JSON and CSV formats.

## 1. Install dependencies

Run in Google Colab. Hugging Face 4-bit inference requires a CUDA GPU; Groq inference does not.

In [1]:
%pip install -q -U "numpy<2.3" pandas "transformers>=4.45" accelerate bitsandbytes gradio groq huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


## 2. Imports, credentials, and defaults

In Colab, add `HF_TOKEN` and/or `GROQ_API_KEY` under Secrets. Outside Colab, set them as environment variables.

In [2]:
import os, re, json, math, tempfile
from datetime import datetime, timezone
from openai import OpenAI
import pandas as pd
import torch
import gradio as gr
from groq import Groq
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")

DEFAULT_HF_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
DEFAULT_GROQ_MODEL = "openai/gpt-oss-20b"

_hf_model = None
_hf_tokenizer = None
_hf_loaded_model_id = None

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 3. Model loading and inference

Both generation and analysis use `call_selected_model()` with the same provider and model ID. Responses are handled internally and are not displayed or logged.

In [9]:
def load_huggingface_model(model_id):
    global _hf_model, _hf_tokenizer, _hf_loaded_model_id

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Hugging Face local 4-bit inference requires a CUDA GPU. "
            "Use a GPU runtime or select Groq."
        )
    if _hf_model is not None and _hf_loaded_model_id == model_id:
        return _hf_model, _hf_tokenizer

    if _hf_model is not None:
        del _hf_model, _hf_tokenizer
        _hf_model = _hf_tokenizer = None
        torch.cuda.empty_cache()

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or None)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, device_map="auto", quantization_config=quant_config,
        torch_dtype=dtype, token=HF_TOKEN or None,
    )
    model.eval()
    _hf_model, _hf_tokenizer, _hf_loaded_model_id = model, tokenizer, model_id
    return model, tokenizer


def call_huggingface(messages, model_id, max_new_tokens=5000):
    model, tokenizer = load_huggingface_model(model_id)
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def call_groq(messages, model_id, max_tokens=6000):
    api_key = GROQ_API_KEY or os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("GROQ_API_KEY is missing. Add it to Colab Secrets or environment.")
    client = Groq(api_key=api_key)
    result = client.chat.completions.create(
        model=model_id, messages=messages, temperature=0.1,
        max_tokens=max_tokens,
    )
    text = result.choices[0].message.content
    if not text:
        raise RuntimeError("The selected Groq model returned an empty response.")
    return text


def call_selected_model(provider, messages, max_new_tokens=1024):
    model_id = get_selected_model(provider)
    print("model id",model_id)

    if provider == "Hugging Face (Local)":
        return call_huggingface(
            model_id=model_id,
            messages=messages,
            max_new_tokens=max_new_tokens
        )

    elif provider == "Groq (Hosted Open-Weight)":
        return call_groq(
            model_id=model_id,
            messages=messages,
            max_tokens=max_new_tokens
        )

    raise ValueError(f"Unsupported provider: {provider}")

In [10]:
def get_selected_model(provider):
    if provider == "Hugging Face (Local)":
        return DEFAULT_HF_MODEL

    elif provider == "Groq (Hosted Open-Weight)":
        return DEFAULT_GROQ_MODEL

    raise ValueError(f"Unsupported provider: {provider}")

## 4. JSON parsing and validation

In [5]:
def extract_json_array(model_text):
    cleaned = model_text.strip()
    cleaned = re.sub(r"^```(?:json)?\\s*", "", cleaned, flags=re.I)
    cleaned = re.sub(r"\\s*```$", "", cleaned)
    start, end = cleaned.find("["), cleaned.rfind("]")
    if start < 0 or end <= start:
        raise ValueError("Model response did not contain a JSON array.")
    result = json.loads(cleaned[start:end + 1])
    if not isinstance(result, list):
        raise ValueError("Expected a JSON array.")
    return result


PROJECT_FIELDS = [
    "project_id", "project_name", "project_type", "planned_duration_days",
    "elapsed_days", "completion_pct", "team_size", "planned_velocity_points",
    "actual_velocity_points", "open_bugs", "critical_bugs",
    "requirement_changes", "blocked_days", "dependency_count",
    "code_review_wait_hours", "team_availability_pct", "scope_change_pct",
    "client_response_delay_days",
]
NUMERIC_FIELDS = PROJECT_FIELDS[3:]


def validate_generated_projects(records, expected_count, batch_number, start_index):
    if len(records) != expected_count:
        raise ValueError(
            f"Batch {batch_number}: model returned {len(records)} projects; "
            f"expected {expected_count}. Reduce the batch size or use a model "
            "with a larger output-token limit."
        )
    result = []
    for offset, item in enumerate(records):
        if not isinstance(item, dict):
            raise ValueError(f"Batch {batch_number}: a project is not a JSON object.")
        missing = [field for field in PROJECT_FIELDS if field not in item]
        if missing:
            raise ValueError(f"Batch {batch_number}: missing fields: {missing}")
        item = dict(item)
        item["project_id"] = f"PRJ-{start_index + offset:04d}"
        for field in NUMERIC_FIELDS:
            try:
                value = float(item[field])
                if not math.isfinite(value):
                    raise ValueError()
                item[field] = int(value) if value.is_integer() else round(value, 2)
            except (TypeError, ValueError):
                raise ValueError(f"Invalid numeric field {field} in {item['project_id']}.")
        result.append(item)
    return result


def normalize_analysis(model_text, projects):
    parsed = extract_json_array(model_text)
    by_id = {
        str(row.get("project_id")): row for row in parsed
        if isinstance(row, dict) and row.get("project_id") is not None
    }
    output = []
    for project in projects:
        item = by_id.get(project["project_id"])
        if item is None:
            raise ValueError(f"Analysis missing {project['project_id']}.")
        probability = float(item["predicted_delay_probability"])
        if not 0 <= probability <= 1:
            raise ValueError(f"Invalid probability for {project['project_id']}.")
        risk = str(item["predicted_delay_risk"]).title()
        if risk not in ("Low", "Medium", "High"):
            raise ValueError(f"Invalid risk category for {project['project_id']}.")
        factors = item.get("contributing_factors", [])
        actions = item.get("recommended_actions", [])
        if not isinstance(factors, list) or not isinstance(actions, list):
            raise ValueError("contributing_factors and recommended_actions must be arrays.")
        analysis = {
            "predicted_delay_probability": round(probability, 3),
            "predicted_delay_risk": risk,
            "predicted_delay_days": max(0, int(item["predicted_delay_days"])),
            "primary_root_cause": str(item.get("primary_root_cause", "")),
            "contributing_factors": factors,
            "recommended_actions": actions,
            "explanation": str(item.get("explanation", "")),
            "model_analysis_status": "Parsed",
        }
        output.append({**project, **analysis})
    return output

## 5. Prompts for LLM-based dataset generation and analysis

The generation prompt creates project metrics only. The analysis prompt then receives those exact records. No random generator or hidden synthetic outcome is used.

In [6]:
def build_generation_messages(count, batch_number, start_index):
    system = '''
You are an experienced software project operations data specialist.
Create realistic, varied demonstration records for software projects.
Return valid JSON only: an array with exactly the requested number of objects.
No Markdown fences or text outside JSON.

Each object must contain:
{
 "project_id": "temporary unique ID",
 "project_name": "short plausible project name",
 "project_type": "Web Application, Mobile Application, Data Platform, API Integration, Enterprise Software, or Cloud Migration",
 "planned_duration_days": integer 20-240,
 "elapsed_days": integer 5-planned_duration_days,
 "completion_pct": number 1-99,
 "team_size": integer 2-25,
 "planned_velocity_points": integer 10-100,
 "actual_velocity_points": number 1-120,
 "open_bugs": integer 0-60,
 "critical_bugs": integer 0-open_bugs,
 "requirement_changes": integer 0-20,
 "blocked_days": integer 0-40,
 "dependency_count": integer 0-20,
 "code_review_wait_hours": number 1-120,
 "team_availability_pct": number 40-100,
 "scope_change_pct": number 0-60,
 "client_response_delay_days": integer 0-30
}
Make records meaningfully varied and broadly internally consistent.
Include a mix of low-risk-looking and high-risk-looking metrics.
Do not include actual outcomes, delay labels, risk scores, causes, or recommendations.
'''
    user = (
        f"Generate exactly {count} project records for batch {batch_number}. "
        f"These will be assigned IDs beginning at {start_index}. "
        "Use temporary unique IDs. Return only the JSON array."
    )
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


def build_analysis_messages(projects):
    system = '''
You are a senior software delivery risk analyst advising a CTO.
Analyze only the supplied metrics. Do not invent facts or claim certainty.
Return valid JSON only: an array with exactly one object per supplied project_id.
No Markdown fences or text outside JSON.

Each object must contain:
{
 "project_id": "same ID as input",
 "predicted_delay_probability": number from 0 to 1,
 "predicted_delay_risk": "Low", "Medium", or "High",
 "predicted_delay_days": non-negative integer estimate,
 "primary_root_cause": "short evidence-based explanation",
 "contributing_factors": ["factor 1", "factor 2"],
 "recommended_actions": ["specific action 1", "specific action 2"],
 "explanation": "brief explanation grounded in supplied metrics"
}
Risk bands: Low < 0.35; Medium >= 0.35 and < 0.70; High >= 0.70.
Consider blocked days, requirement changes, bugs, critical bugs, dependencies,
velocity gap, availability, scope change, review wait, and client response delay.
Treat estimates as uncertain. Do not change IDs or infer actual outcomes.
'''
    user = "Analyze these project records and return the required JSON array:\n"
    user += json.dumps(projects, ensure_ascii=False)
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]

## 6. Batch processing

For 20 projects with batch size 5, the app processes four batches. Each batch makes two calls to the same selected model: one to generate records and one to analyze them.

In [7]:
def dataframe_for_display(records):
    df = pd.DataFrame(records)
    for col in ("contributing_factors", "recommended_actions"):
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False)
                if isinstance(x, (list, dict)) else x
            )
    return df


def generate_dataset(provider, n_projects, batch_size):
    try:
        n_projects, batch_size = int(n_projects), int(batch_size)
        model_id = get_selected_model(provider)
        if not 1 <= n_projects <= 200:
            raise ValueError("Choose between 1 and 200 projects.")
        if not 1 <= batch_size <= 10:
            raise ValueError("Batch size must be between 1 and 10.")
        if not model_id:
            raise ValueError("Enter a model ID.")

        total_batches = math.ceil(n_projects / batch_size)
        records = []
        next_id = 1
        yield (f"Starting {n_projects} projects in {total_batches} batches...",
               dataframe_for_display(records), None, None)

        for batch_no in range(1, total_batches + 1):
            count = min(batch_size, n_projects - len(records))
            progress = f"Batch {batch_no}/{total_batches}"

            # Step 1: selected LLM generates project data.
            yield (f"{progress}: generating {count} project records...",
                   dataframe_for_display(records), None, None)
            gen_messages = build_generation_messages(count, batch_no, next_id)
            generated_text = call_selected_model(
                provider, gen_messages, max_new_tokens=5000
            )
            generated = validate_generated_projects(
                extract_json_array(generated_text), count, batch_no, next_id
            )

            # Step 2: same provider and model analyzes those generated records.
            yield (f"{progress}: generated records; analyzing with the same model...",
                   dataframe_for_display(records), None, None)
            analysis_messages = build_analysis_messages(generated)
            analysis_text = call_selected_model(
                provider, analysis_messages, max_new_tokens=5000
            )
            analyzed = normalize_analysis(analysis_text, generated)
            records.extend(analyzed)
            next_id += count

            yield (f"Completed {progress}: {len(records)}/{n_projects} projects.",
                   dataframe_for_display(records), None, None)

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        folder = tempfile.mkdtemp(prefix="project_delivery_delay_")
        json_path = os.path.join(folder, f"project_delivery_analysis_{stamp}.json")
        csv_path = os.path.join(folder, f"project_delivery_analysis_{stamp}.csv")

        payload = {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "problem": "Project delivery delay risk analysis and root-cause assessment",
            "data_type": "LLM-generated demonstration project data",
            "provider": provider,
            "model_id": model_id,
            "number_of_projects": len(records),
            "generation_method": "Selected LLM generated each project batch",
            "analysis_method": "The same selected LLM analyzed each generated batch",
            "disclaimer": "LLM-generated demonstration data and estimates; not validated predictions or actual outcomes.",
            "records": records,
        }
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        dataframe_for_display(records).to_csv(
            csv_path, index=False, encoding="utf-8-sig"
        )

        yield (f"Finished: {len(records)} projects generated and analyzed.",
               dataframe_for_display(records), json_path, csv_path)

    except Exception as exc:
        yield (f"Error: {type(exc).__name__}: {exc}",
               pd.DataFrame(), None, None)

## 7. Gradio interface

The interface includes provider/model selection, project count, batch size, status, dataset preview, and JSON/CSV downloads. Raw model output and random seed controls are removed.

In [11]:
with gr.Blocks(title="Project Delivery Delay Intelligence") as demo:
    gr.Markdown(
        "# Project Delivery Delay Prediction & Root-Cause Analysis\n"
        "Generate project records and analyze delivery risks with the same selected LLM."
    )
    with gr.Row():
        provider = gr.Dropdown(
            choices=["Hugging Face (Local)", "Groq (Hosted Open-Weight)"],
            value="Groq (Hosted Open-Weight)", label="AI Provider"
        )

    with gr.Row():
        n_projects = gr.Slider(
            1, 100, value=20, step=1, label="Number of projects"
        )
        batch_size = gr.Slider(
            1, 10, value=5, step=1, label="Projects per batch"
        )

    generate_btn = gr.Button("Generate and Analyze Dataset", variant="primary")
    status = gr.Markdown("Choose a provider/model and click Generate.")
    preview = gr.Dataframe(
        label="Generated projects and AI analysis", interactive=False, wrap=True
    )
    with gr.Row():
        json_download = gr.File(label="Download JSON dataset")
        csv_download = gr.File(label="Download CSV dataset")

    generate_btn.click(
        fn=generate_dataset,
        inputs=[provider, n_projects, batch_size],
        outputs=[status, preview, json_download, csv_download],
    )
    gr.Markdown(
        "**Notes:** Hugging Face local 4-bit inference requires a CUDA GPU and "
        "may require authorized access to gated models. Groq requires "
        "`GROQ_API_KEY` and a currently supported model ID. Both stages use "
        "the provider/model selected above. Review all AI-generated estimates."
    )

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5d889578932e656749.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


model id openai/gpt-oss-20b
model id openai/gpt-oss-20b
model id openai/gpt-oss-20b
model id openai/gpt-oss-20b
model id openai/gpt-oss-20b
model id openai/gpt-oss-20b
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5d889578932e656749.gradio.live


## 8. Running notes and troubleshooting

- **Groq:** Add `GROQ_API_KEY` to Colab Secrets. Replace the default model ID if it is not available in your Groq account.
- **Hugging Face:** Select a GPU runtime, add `HF_TOKEN` if required, and choose a compatible causal language model with a chat template.
- **Incomplete JSON:** Reduce batch size (for example, from 5 to 2 or 3). Larger batches require more output tokens.
- **Provider model IDs are different:** A Hugging Face repository ID is not necessarily a valid Groq model ID.
- **Interpretation:** The risk score is the selected LLM's assessment of generated metrics. This is not a trained predictive model and is not evidence of actual project delay.